# ДЗ 2.2  Задание 2

Соединяем просмотры с каталогом фильмов по element_uid. Сначала определяем, какие ключи встречаются особенно часто и как это влияет на выполнение Spark. Затем сравниваем три варианта: исходный SortMergeJoin, рассылку небольшого каталога (BroadcastHashJoin) и Salt Join для популярных ключей.

## 1. Подключение Spark и подготовка файлов


In [1]:
import os
import random
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, LongType

from hw22_utils import (
    BASE_DIR, S3_ROOT, ensure_data, flatten_catalogue, configure_s3,
    physical_plan, save_plan, write_and_measure, save_measurements, output_file_count,
    spark_ui_status,
)

spark = (SparkSession.builder.appName('DWH_HW2_2_Task2_Skew')
         .master('local[4]')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
configure_s3(spark)
print('Версия Spark:', spark.version)
assert spark.version.startswith('3.5.')

folder = ensure_data('task2')
cat_flat = flatten_catalogue(folder / 'catalogue.json')
print('Каталог:', cat_flat)
print('Просмотры:', folder / 'skew_transactions.csv')

Версия Spark: 3.5.0
Каталог: /home/jovyan/work/hw2_2/data4hw2_2/task2/catalogue_rows.jsonl
Просмотры: /home/jovyan/work/hw2_2/data4hw2_2/task2/skew_transactions.csv


## 2. Чтение и проверка исходных таблиц

Читаем csv с явными числовыми типами. Проверяем пустые идентификаторы и повторяющиеся ключи.

Для всех вариантов оставляем одинаковый набор колонок каталога - type, duration, feature_1

In [2]:
tx_schema = StructType([
    StructField('user_uid', LongType(), True),
    StructField('element_uid', LongType(), True),
    StructField('watched_time', LongType(), True),
])
watch = (spark.read.option('header', True).schema(tx_schema)
         .csv(str(folder / 'skew_transactions.csv')))
catalogue = (spark.read.json(str(cat_flat))
             .select(F.col('element_uid').cast('long').alias('element_uid'),
                     'type', 'duration', 'feature_1'))

print('Просмотров:', watch.count(), 'фильмов в каталоге:', catalogue.count())
watch.show(4, truncate=False)
catalogue.show(4, truncate=False)
null_watch = watch.filter('element_uid IS NULL').count()
null_cat = catalogue.filter('element_uid IS NULL').count()
dup_cat = catalogue.groupBy('element_uid').count().filter('count > 1').count()
print('Пустых ключей просмотров:', null_watch)
print('Пустых ключей каталога:', null_cat)
print('Повторяющихся ключей каталога:', dup_cat)
assert null_cat == 0 and dup_cat == 0, 'Каталог не имеет уникального ключа'

Просмотров: 30070692 фильмов в каталоге: 10200
+--------+-----------+------------+
|user_uid|element_uid|watched_time|
+--------+-----------+------------+
|408794  |2714       |7583        |
|247546  |2714       |4450        |
|43556   |2714       |7569        |
|517299  |2714       |1           |
+--------+-----------+------------+
only showing top 4 rows

+-----------+-----+--------+--------------------+
|element_uid|type |duration|feature_1           |
+-----------+-----+--------+--------------------+
|1983       |movie|140     |1657223.396513469   |
|3783       |movie|110     |3.5565207694893226E7|
|5208       |movie|90      |1.327067652431015E7 |
|9744       |movie|120     |2.1749917409822803E7|
+-----------+-----+--------+--------------------+
only showing top 4 rows

Пустых ключей просмотров: 0
Пустых ключей каталога: 0
Повторяющихся ключей каталога: 0


## 3. Диагностика перекоса

Подсчитываем, сколько раз встречается каждый element_uid.

Если несколько фильмов сосредоточили большую долю строк(перекос), то при обычном соединении отдельные задачи получают значительно больше работы. Наличие перекоса подтверждено также длительностью задач в Spark UI.

Независимая статистика по полному CSV: 30 070 692 просмотра, 8296 разных идентификаторов фильмов, из них 20 496 000 приходятся на element_uid = 2714. (файл task2_key_profile.csv)

In [ ]:
non_null_watch = watch.filter('element_uid IS NOT NULL')
watch_count = non_null_watch.count()
frequencies = (non_null_watch.groupBy('element_uid').count()
               .withColumn('share', F.col('count') / F.lit(watch_count))
               .orderBy(F.desc('count')))
frequencies.show(15, truncate=False)
high = frequencies.limit(10).collect()
print('Доля крупнейшего ключа:', f'{high[0]["share"]:.2%}')
print('Совокупная доля 10 самых частых:', f'{sum(row["share"] for row in high):.2%}')
HOT_THRESHOLD = 0.01
hot_ids = [int(row['element_uid']) for row in high if row['share'] >= HOT_THRESHOLD]
if not hot_ids:
    hot_ids = [int(high[0]['element_uid'])]
print('Ключи для распределения Salt Join:', hot_ids)
assert watch_count == 30_070_692
assert int(high[0]['element_uid']) == 2714 and int(high[0]['count']) == 20_496_000
print('Сравнение с контрольными значениями полного CSV пройдено.')

+-----------+--------+---------------------+
|element_uid|count   |share                |
+-----------+--------+---------------------+
|2714       |20496000|0.6815938921525317   |
|747        |61272   |0.0020375986026527093|
|2639       |61119   |0.002032510592040915 |
|3783       |55228   |0.0018366055560011722|
|2245       |52942   |0.001760584691566127 |
|6127       |52168   |0.0017348453437652847|
|10061      |48993   |0.001629260809827722 |
|3916       |48243   |0.0016043195813385338|
|8771       |47480   |0.0015789460382221998|
|3336       |45646   |0.0015179564208233052|
|7079       |43872   |0.0014589621017035458|
|3567       |42278   |0.0014059536774211913|
|6195       |42192   |0.0014030937498877645|
|4141       |41704   |0.0013868653238841328|
|1016       |41539   |0.0013813782536165114|
+-----------+--------+---------------------+
only showing top 15 rows

Доля крупнейшего ключа: 68.16%
Совокупная доля 10 самых частых: 69.73%
Ключи для распределения Salt Join: [2714]
Сравне

## 4. INNER JOIN с перераспределением по ключу

Для исходного опыта отключаем автоматическую рассылку маленького каталога и адаптивное выполнение AQE.

Проверяем по плану, что используется SortMergeJoin. Записываем строки в Parquet в RustFS.

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
spark.conf.set('spark.sql.shuffle.partitions', '16')

COLS = ['user_uid', 'element_uid', 'watched_time', 'type', 'duration', 'feature_1']

def baseline_join():
    return watch.join(catalogue, on='element_uid', how='inner').select(*COLS)

baseline = baseline_join()
print(physical_plan(baseline)[:2500])
save_plan(baseline, 'task2_plan_baseline.txt')
assert 'SortMergeJoin' in physical_plan(baseline), 'Ожидался SortMergeJoin'
print('Исходный план содержит SortMergeJoin.')

*(5) Project [user_uid#0L, element_uid#1L, watched_time#2L, type#23, duration#16L, feature_1#18]
+- *(5) SortMergeJoin [element_uid#1L], [element_uid#17L], Inner
   :- *(2) Sort [element_uid#1L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(element_uid#1L, 16), ENSURE_REQUIREMENTS, [plan_id=508]
   :     +- *(1) Filter isnotnull(element_uid#1L)
   :        +- FileScan csv [user_uid#0L,element_uid#1L,watched_time#2L] Batched: false, DataFilters: [isnotnull(element_uid#1L)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/hw2_2/data4hw2_2/task2/skew_transactions.csv], PartitionFilters: [], PushedFilters: [IsNotNull(element_uid)], ReadSchema: struct<user_uid:bigint,element_uid:bigint,watched_time:bigint>
   +- *(4) Sort [element_uid#17L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(element_uid#17L, 16), ENSURE_REQUIREMENTS, [plan_id=516]
         +- *(3) Filter isnotnull(element_uid#17L)
            +- FileScan json [duration#16L,ele

## 5. Оптимизация рассылкой каталога

Каталог намного меньше таблицы просмотров, поэтому его можно скопировать каждому исполнителю. В этом случае большой набор просмотров не приходится перераспределять по популярному ключу.ъ

Используем те же входные строки и выходные колонки, меняется только выбор алгоритма JOIN.

In [5]:
def broadcast_join():
    return watch.join(F.broadcast(catalogue), on='element_uid', how='inner').select(*COLS)

broadcast_result = broadcast_join()
print(physical_plan(broadcast_result)[:2500])
save_plan(broadcast_result, 'task2_plan_broadcast.txt')
assert 'BroadcastHashJoin' in physical_plan(broadcast_result)
print('План содержит BroadcastHashJoin.')

*(2) Project [user_uid#0L, element_uid#1L, watched_time#2L, type#23, duration#16L, feature_1#18]
+- *(2) BroadcastHashJoin [element_uid#1L], [element_uid#17L], Inner, BuildRight, false
   :- *(2) Filter isnotnull(element_uid#1L)
   :  +- FileScan csv [user_uid#0L,element_uid#1L,watched_time#2L] Batched: false, DataFilters: [isnotnull(element_uid#1L)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/hw2_2/data4hw2_2/task2/skew_transactions.csv], PartitionFilters: [], PushedFilters: [IsNotNull(element_uid)], ReadSchema: struct<user_uid:bigint,element_uid:bigint,watched_time:bigint>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, bigint, false]),false), [plan_id=571]
      +- *(1) Filter isnotnull(element_uid#17L)
         +- FileScan json [duration#16L,element_uid#17L,feature_1#18,type#23] Batched: false, DataFilters: [isnotnull(element_uid#17L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/hw2_2/data4hw2_2/task2/ca

## 6. Salt Join

Для популярных фильмов вычисляем дополнительное число salt от 0 до SALT_BUCKETS - 1 на стороне просмотров. Для остальных фильмов salt = 0.

В результате каждый исходный просмотр по прежнему соединяется с ровно одной копией соответствующего фильма. После соединения колонка _salt отбрасывается.

Здесь выбрано 16 значений salt - число разделов для перераспределения. 

In [ ]:
SALT_BUCKETS = 16
assert SALT_BUCKETS >= 2

watch_salted = watch.withColumn(
    '_salt',
    F.when(F.col('element_uid').isin(hot_ids),
           F.pmod(F.xxhash64('user_uid', 'element_uid', 'watched_time'),
                   F.lit(SALT_BUCKETS)).cast('int'))
     .otherwise(F.lit(0))
)

catalogue_salted = (catalogue
    .withColumn('_salts',
                F.when(F.col('element_uid').isin(hot_ids),
                       F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1)))
                 .otherwise(F.array(F.lit(0))))
    .withColumn('_salt', F.explode('_salts'))
    .drop('_salts'))


def salt_join():
    return (watch_salted.hint('MERGE')
            .join(catalogue_salted.hint('MERGE'),
                  on=['element_uid', '_salt'], how='inner')
            .select(*COLS))

salt_result = salt_join()
print(physical_plan(salt_result)[:3000])
save_plan(salt_result, 'task2_plan_salt.txt')
assert 'SortMergeJoin' in physical_plan(salt_result)
print('Выделенных популярных ключей:', len(hot_ids))
print('Число копий каждого популярного фильма:', SALT_BUCKETS)
print('Дополнительных строк каталога:', len(hot_ids) * (SALT_BUCKETS - 1))

*(5) Project [user_uid#0L, element_uid#1L, watched_time#2L, type#23, duration#16L, feature_1#18]
+- *(5) SortMergeJoin [element_uid#1L, _salt#193], [element_uid#17L, _salt#205], Inner
   :- *(2) Sort [element_uid#1L ASC NULLS FIRST, _salt#193 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(element_uid#1L, _salt#193, 16), ENSURE_REQUIREMENTS, [plan_id=620]
   :     +- *(1) Project [user_uid#0L, element_uid#1L, watched_time#2L, CASE WHEN (element_uid#1L = 2714) THEN cast(pmod(xxhash64(user_uid#0L, element_uid#1L, watched_time#2L, 42), 16) as int) ELSE 0 END AS _salt#193]
   :        +- *(1) Filter (isnotnull(element_uid#1L) AND CASE WHEN (element_uid#1L = 2714) THEN isnotnull(cast(pmod(xxhash64(user_uid#0L, element_uid#1L, watched_time#2L, 42), 16) as int)) ELSE true END)
   :           +- FileScan csv [user_uid#0L,element_uid#1L,watched_time#2L] Batched: false, DataFilters: [isnotnull(element_uid#1L), CASE WHEN (element_uid#1L = 2714) THEN isnotnull(cast(pmod(xxhash64(u...

## 7. Измерение времени и распределения нагрузки

Для исходного соединения, рассылки каталога и Salt Join выполняем один пробный запуск (прогрев) и по три измерения полной записи результата в Parquet в S3. Порядок вариантов чередуется.

Фиксируем время, объём перераспределения (shuffle), выгрузку данных из памяти (spill) и длительность отдельных задач Spark. Статистика по этапам сохраняется в results/task2_stages.csv

In [ ]:
RUN_BENCHMARKS = True         # False: открыть сохранённые таблицы без повторной записи
REQUIRE_UI_METRICS = True     # Не начинать дорогостоящие замеры, если Spark UI недоступен
STRATEGIES = {'baseline': baseline_join,
              'broadcast': broadcast_join,
              'salt': salt_join}
TARGETS = {name: f'{S3_ROOT}/task2/results/{name}/' for name in STRATEGIES}
REPEATS = 3

if RUN_BENCHMARKS:
    ui = spark_ui_status(spark)
    print('Spark UI:', ui)
    if REQUIRE_UI_METRICS and not ui['ok']:
        raise RuntimeError('Spark UI REST недоступен; до измерений проверьте окружение')
    records = []
    for cycle in range(REPEATS + 1):
        order = list(STRATEGIES)
        random.Random(121 + cycle).shuffle(order)
        for name in order:
            record = write_and_measure(spark, STRATEGIES[name](), TARGETS[name],
                                       name, cycle, warmup=(cycle == 0))
            record['shuffle_partitions'] = spark.conf.get('spark.sql.shuffle.partitions')
            records.append(record)
            print(('Прогрев' if cycle == 0 else f'Замер {cycle}'),
                  name, f"{record['seconds']:.3f} с", '| метрики:', record['metrics_status'])
    raw, summary = save_measurements(records, 'task2')
else:
    raw = pd.read_csv(BASE_DIR / 'results' / 'task2_measurements.csv')
    summary = pd.read_csv(BASE_DIR / 'results' / 'task2_summary.csv')
    records = raw.to_dict('records')

if 'metrics_status' not in raw.columns:
    raw['metrics_status'] = 'Не собрано прежней версией скрипта'

display(raw[['strategy', 'round', 'warmup', 'seconds', 'shuffle_read_bytes',
             'shuffle_write_bytes', 'memory_spilled_bytes', 'disk_spilled_bytes',
             'task_duration_max_median', 'metrics_status']])
display(summary.sort_values('median_seconds'))
missing = raw.loc[raw['shuffle_read_bytes'].isna(),
                  ['strategy', 'round', 'metrics_status']]
if len(missing):
    print('ВНИМАНИЕ: метрики Spark UI не получены в следующих замерах:')
    display(missing)
else:
    print('Метрики всех записей получены. Детали по этапам — results/task2_stages.csv')


Spark UI: {'ok': True, 'api_url': 'http://127.0.0.1:4041/api/v1', 'application_id': 'local-1794573454728', 'message': 'Доступен: 30 этапов зарегистрировано'}
Прогрев broadcast 56.433 с | метрики: ok
Прогрев salt 35.947 с | метрики: ok
Прогрев baseline 46.472 с | метрики: ok
Замер 1 broadcast 27.448 с | метрики: ok
Замер 1 baseline 37.201 с | метрики: ok
Замер 1 salt 40.727 с | метрики: ok
Замер 2 salt 43.805 с | метрики: ok
Замер 2 broadcast 26.936 с | метрики: ok
Замер 2 baseline 51.191 с | метрики: ok
Замер 3 salt 51.254 с | метрики: ok
Замер 3 baseline 33.998 с | метрики: ok
Замер 3 broadcast 39.935 с | метрики: ok


,strategy,round,warmup,seconds,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,task_duration_max_median,metrics_status
0,broadcast,0,True,56.433,0,0,0,0,1.00,ok
1,salt,0,True,35.947,393365534,393365534,943714800,244614569,2.65,ok
2,baseline,0,True,46.472,376869985,376869985,742388976,233282762,3.77,ok
3,broadcast,1,False,27.448,0,0,0,0,1.42,ok
4,baseline,1,False,37.201,376869985,376869985,763360416,243019005,4.23,ok
5,salt,1,False,40.727,393365534,393365534,943714800,245058750,3.27,ok
6,salt,2,False,43.805,393365534,393365534,1006629120,261218324,1.91,ok
7,broadcast,2,False,26.936,0,0,0,0,1.54,ok
8,baseline,2,False,51.191,376869985,376869985,746583264,236010365,8.33,ok
9,salt,3,False,51.254,393365534,393365534,964686240,250586305,2.98,ok


,strategy,runs,median_seconds,min_seconds,max_seconds,shuffle_read_bytes_median,shuffle_write_bytes_median,memory_spilled_bytes_max,disk_spilled_bytes_max,task_duration_max_median_max
1,broadcast,3,27.448,26.936,39.935,0.0,0.0,0.000000e+00,0.0,1.54
0,baseline,3,37.201,33.998,51.191,376869985.0,376869985.0,7.633604e+08,243019005.0,8.33
2,salt,3,43.805,40.727,51.254,393365534.0,393365534.0,1.006629e+09,261218324.0,3.27


Метрики всех записей получены. Детали по этапам — results/task2_stages.csv


### 7.1. Проверяем разброс времени выполнения задач Spark

Каждый этап состоит из отдельных параллельных задач. Если одна из них работает значительно дольше медианной задачи того же этапа, это признак неравномерной нагрузки.

Для корректного сравнения показываем отдельные этапы, а не объединяем длительности задач разных этапов в одну медиану. При сборе метрик их перечень уже сохранён в results/task2_stages.csv. Если этот файл пуст, нужно проверить Spark UI.

In [8]:
stage_file = BASE_DIR / 'results' / 'task2_stages.csv'
if not stage_file.exists():
    print('Файл этапов отсутствует: прежняя версия не собирала их. Нужны новые замеры.')
    stages = pd.DataFrame(columns=['warmup', 'task_max_median_ratio', 'num_tasks'])
else:
    stages = pd.read_csv(stage_file)
stages = stages.loc[~stages['warmup'].astype(bool)].copy()
valid = stages.loc[stages['task_max_median_ratio'].notna() &
                   (stages['num_tasks'] >= 2)].copy()
if len(valid):
    display(valid.sort_values('task_max_median_ratio', ascending=False)[
        ['strategy', 'round', 'stage_id', 'stage_name', 'num_tasks',
         'task_median_ms', 'task_max_ms', 'task_max_median_ratio',
         'shuffle_read_bytes', 'disk_spilled_bytes']].head(20))
    print('Отношение максимум/медиана считается внутри одного этапа Spark.')
else:
    print('Длительности отдельных задач не получены. Посмотрите Spark UI → Stages → Tasks.')

,strategy,round,stage_id,stage_name,num_tasks,task_median_ms,task_max_ms,task_max_median_ratio,shuffle_read_bytes,disk_spilled_bytes
21,baseline,2,53,parquet at NativeMethodAccessorImpl.java:0,16,4242.5,35361.0,8.33,376869985,236010365
10,baseline,1,42,parquet at NativeMethodAccessorImpl.java:0,16,3702.0,15646.0,4.23,376869985,243019005
27,baseline,3,59,parquet at NativeMethodAccessorImpl.java:0,16,5014.5,16892.0,3.37,376869985,211456171
13,salt,1,45,parquet at NativeMethodAccessorImpl.java:0,16,1865.5,6091.0,3.27,393365534,245058750
24,salt,3,56,parquet at NativeMethodAccessorImpl.java:0,16,2356.5,7014.0,2.98,393365534,250586305
16,salt,2,48,parquet at NativeMethodAccessorImpl.java:0,16,7733.5,14788.0,1.91,393365534,261218324
19,broadcast,2,50,parquet at NativeMethodAccessorImpl.java:0,4,10197.5,15698.0,1.54,0,0
8,broadcast,1,39,parquet at NativeMethodAccessorImpl.java:0,4,12166.0,17309.0,1.42,0,0
11,baseline,1,41,parquet at NativeMethodAccessorImpl.java:0,4,12418.5,12797.0,1.03,0,0
28,baseline,3,58,parquet at NativeMethodAccessorImpl.java:0,4,7185.0,7375.0,1.03,0,0


Отношение максимум/медиана считается внутри одного этапа Spark.


## 8. Проверяем, что все совпадения сохранены

Нельзя проверять только одинаковое число строк: разные результаты могут иметь одинаковый размер. Сравниваем каждый вариант с исходным в обоих направлениях через exceptAll - учитываются повторы. В каждой таблице также проверяем число частей Parquet.

In [9]:
results = {name: spark.read.parquet(path) for name, path in TARGETS.items()}
reference = results['baseline']
checks = []
for name, df in results.items():
    checks.append({
        'strategy': name,
        'rows': df.count(),
        'missing': reference.exceptAll(df).count(),
        'extra': df.exceptAll(reference).count(),
        'parquet_files': output_file_count(TARGETS[name]),
    })
checks_df = pd.DataFrame(checks)
display(checks_df)
assert (checks_df[['missing', 'extra']] == 0).all().all()
assert (checks_df['rows'] > 0).all()
checks_df.to_csv(BASE_DIR / 'results' / 'task2_correctness.csv', index=False)
print('Все три результата полностью совпадают, включая повторяющиеся строки.')

,strategy,rows,missing,extra,parquet_files
0,baseline,30070692,0,0,16
1,broadcast,30070692,0,0,4
2,salt,30070692,0,0,16


Все три результата полностью совпадают, включая повторяющиеся строки.


## 9. Выводы по заданию 2

Из 30 070 692 просмотров 20 496 000 (68,16%) относятся к фильму element_uid = 2714. При соединении с перераспределением по element_uid строки этого фильма могут сосредоточиться в одной задаче. Пустых ключей нет, в каталоге 10 200 уникальных фильмов, повторяющихся ключей не обнаружено.

Сравниваемые способы:
- SortMergeJoin перераспределяет обе таблицы по идентификатору фильма.
- BroadcastHashJoin рассылает небольшой каталог рабочим процессам и не перераспределяет крупную таблицу просмотров по ключу соединения. Способ применим, пока каталог помещается в доступную память.
- Salt Join распределяет просмотры самого частого фильма по 16 дополнительным значениям ключа и создаёт 16 копий его записи в каталоге. Для остальных фильмов копии не создаются. Каталог увеличился всего на 15 строк. Каждая строка просмотра получает одно дополнительное значение ключа, поэтому совпадения не теряются и не дублируются.

Broadcast Hash Join: медиана - 27,448 с, минимум - 26,936 с, максимум - 39,935 с.



Исходный Sort Merge Join: медиана - 37,201 с, минимум - 33,998 с, максимум - 51,191 с.



Salt Join: медиана - 43,805 с, минимум - 40,727 с, максимум - 51,254 с.

Производительность: самым быстрым по медиане оказался BroadcastHashJoin- полная запись заняла примерно на 26,2% меньше времени, чем у исходного способа. Salt Join потребовал на 17,8% больше временим чем исходный способ, несмотря на более равномерное распределение задач.

Показатели Spark: для исходного соединения зафиксировано около 376 МБ перераспределённых данных на чтение и запись, для Salt Join - 393 МБ, а для Broadcast 0 МБ. В исходном соединении отношение максимальной длительности задачи к медианной в одном из этапов достигало 8,33, с Salt Join - 3,27. Таким образом, разделение популярного ключа уменьшило неравномерность длительности задач, но добавило вычисления и не ускорило всю запись. У исходного соединения и Salt Join также наблюдался spill, тогда как у Broadcast он отсутствовал. Максимальные значения spill приведены  выше.

Правильность: все три способа сохранили по 30 070 692 строки,  сравнение через exceptAll не выявило ни потерянных, ни лишних записей, включая повторы. Результаты всех способов совпадают.